In [2]:
# ЭТАП W1. Загрузка данных (только по 26.07.2026 включительно) и матрицы «серия × день»
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CANDIDATES = [
    ROOT / "data" / "processed" / "base_grid.csv",
    ROOT / "data" / "raw" / "base_grid.csv",
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Не найден base_grid.csv. Положи его в data/processed/")

CUT_DATE = pd.Timestamp("2026-07-26")   # последний день, который разрешено использовать (воскресенье)
WEEKS = 9
HORIZON = WEEKS * 7                     # 63 дня
t0 = time.time()

dtypes = {
    "warehouse_name": "category", "article": "category",
    "stock": "float32", "quantity": "float32",
    "cities_name": "category",
    "category_lvl_1": "category", "category_lvl_2": "category", "category_lvl_3": "category",
}
df = pd.read_csv(DATA_PATH, dtype=dtypes, parse_dates=["date"])
df = df[df["date"] <= CUT_DATE]                       # данные после 26.07 использовать нельзя
df = df.sort_values(["warehouse_name", "article", "date"]).reset_index(drop=True)

KEYS = ["warehouse_name", "article"]
df["sid"] = df.groupby(KEYS, observed=True).ngroup()
S = df["sid"].nunique()

dates = pd.date_range(df["date"].min(), df["date"].max())
D = len(dates)
assert dates[-1] == CUT_DATE and CUT_DATE.weekday() == 6, "Данные должны заканчиваться воскресеньем 2026-07-26"
day_idx = (df["date"] - dates[0]).dt.days.values

Q = np.full((S, D), np.nan, dtype=np.float32)
Q[df["sid"].values, day_idx] = df["quantity"].values
ST = np.full((S, D), np.nan, dtype=np.float32)
ST[df["sid"].values, day_idx] = df["stock"].values

Q0 = np.nan_to_num(Q)
first_idx = np.argmax(~np.isnan(Q), axis=1)

meta = df.drop_duplicates("sid").set_index("sid").sort_index()
art_cat = (
    df[df["category_lvl_1"] != "unknown"]
    .drop_duplicates("article")
    .set_index("article")["category_lvl_1"]
)
meta["cat1"] = meta["article"].map(art_cat).astype("object").fillna("unknown")
wh_code = meta["warehouse_name"].cat.codes.values
cat_code = pd.Categorical(meta["cat1"]).codes
del df

print(f"Файл: {DATA_PATH}")
print(f"Серий: {S}, дней: {D}, период: {dates[0].date()} — {dates[-1].date()} ({dates[-1].day_name()})")
print(f"Прогноз: {WEEKS} недель, с {(dates[-1] + pd.Timedelta(days=1)).date()} "
      f"по {(dates[-1] + pd.Timedelta(days=HORIZON)).date()}")
print(f"Отрицательных остатков: {int((ST < 0).sum())}, пропусков остатка: {np.isnan(ST).mean():.1%}")
print(f"Готово за {time.time() - t0:.1f} c")

Файл: C:\Users\nikit\sber-forecast\data\processed\base_grid.csv
Серий: 8789, дней: 957, период: 2023-12-13 — 2026-07-26 (Sunday)
Прогноз: 9 недель, с 2026-07-27 по 2026-09-27
Отрицательных остатков: 29, пропусков остатка: 42.2%
Готово за 30.0 c


In [3]:
# ЭТАП W2. Метрики по новому score.py, недельные окна, базовые прогнозы
def compute_metrics(Y, P):
    """Y, P: (серии x 9 недель). SMAPE: пары 0/0 остаются в среднем с ошибкой 0 (новый score.py)."""
    P = np.clip(P, 0, None)
    den = (np.abs(Y) + np.abs(P)) / 2
    ratio = np.zeros_like(den, dtype=float)
    np.divide(np.abs(Y - P), den, out=ratio, where=den > 0)
    smape = ratio.mean(axis=1).mean() * 100
    denom = np.abs(Y).sum()
    num = np.abs(Y - P).sum()
    wape = (0.0 if num == 0 else 100.0) if denom == 0 else num / denom * 100
    rmse = np.sqrt(((Y - P) ** 2).mean())
    rmsle = np.sqrt(((np.log1p(P) - np.log1p(Y)) ** 2).mean())
    my, mp = Y.mean(axis=1), P.mean(axis=1)
    ok = my > 0
    bias = (np.abs(mp[ok] - my[ok]) / my[ok]).mean() * 100
    return {"SMAPE": smape, "WAPE": wape, "RMSE": rmse, "Bias": bias, "RMSLE": rmsle}


WEIGHTS = {"SMAPE": 0.50, "WAPE": 0.20, "RMSE": 0.15, "Bias": 0.10, "RMSLE": 0.05}


def proxy_score(m, naive):
    """Ориентир (не рейтинг): метрики, делённые на метрики «последней недели», с весами."""
    return sum(WEIGHTS[k] * min(m[k] / naive[k], 1.0) for k in WEIGHTS)


def window_cut(k):
    """Последний известный день окна проверки k (воскресенье), после него идут 9 недель факта."""
    return D - 1 - HORIZON * (k + 1)


def truth(c):
    """Факт: недельные суммы за 9 недель после дня c -> (серии x 9)."""
    return Q0[:, c + 1:c + 1 + HORIZON].reshape(S, WEEKS, 7).sum(axis=2)


def flat_w(rate):
    """Одно число на серию -> плоский прогноз на 9 недель."""
    return np.repeat(np.asarray(rate, dtype=np.float32)[:, None], WEEKS, axis=1)


def baselines(c):
    last4 = np.stack([Q0[:, c - 7 * (i + 1) + 1:c - 7 * i + 1].sum(axis=1) for i in range(4)], axis=1)
    m13 = Q0[:, c - 90:c + 1].sum(axis=1) / 13
    return {
        "нули": flat_w(np.zeros(S)),
        "последняя неделя": flat_w(last4[:, 0]),
        "среднее 4 нед.": flat_w(last4.mean(axis=1)),
        "медиана 4 нед.": flat_w(np.median(last4, axis=1)),
        "среднее 13 нед.": flat_w(m13),
        "среднее 13 нед., ноль если < 0.5": flat_w(np.where(m13 < 0.5, 0, m13)),
    }


N_WINDOWS = 3
naive_m, truths, rows = {}, {}, []
for k in range(N_WINDOWS):
    c = window_cut(k)
    truths[k] = truth(c)
    naive_m[k] = compute_metrics(truths[k], baselines(c)["последняя неделя"])
    for name, P in baselines(c).items():
        m = compute_metrics(truths[k], P)
        rows.append({"подход": name, **m, "score": proxy_score(m, naive_m[k])})

summary = pd.DataFrame(rows).groupby("подход").mean().round(3).sort_values("score")
for k in range(N_WINDOWS):
    c = window_cut(k)
    print(f"окно {k}: до {dates[c].date()}, факт {dates[c + 1].date()} — {dates[c + HORIZON].date()}, "
          f"серий без продаж за 9 недель: {(truths[k].sum(1) == 0).sum()}")
display(summary)

окно 0: до 2026-05-24, факт 2026-05-25 — 2026-07-26, серий без продаж за 9 недель: 1058
окно 1: до 2026-03-22, факт 2026-03-23 — 2026-05-24, серий без продаж за 9 недель: 978
окно 2: до 2026-01-18, факт 2026-01-19 — 2026-03-22, серий без продаж за 9 недель: 877


,SMAPE,WAPE,RMSE,Bias,RMSLE,score
подход,,,,,,
"среднее 13 нед., ноль если < 0.5",70.609,50.412998,2.880,75.769997,0.505,0.857
среднее 13 нед.,123.279,52.296001,2.873,63.303001,0.492,0.892
медиана 4 нед.,75.086,51.394001,2.907,81.416000,0.529,0.898
среднее 4 нед.,98.802,52.234001,2.850,78.639000,0.514,0.905
нули,97.498,100.000000,7.319,100.000000,1.094,0.982
последняя неделя,76.778,60.660999,3.357,122.698997,0.623,1.000


In [4]:
# ЭТАП W3. Недельная модель: признаки, обучающая выборка, параметры
import importlib
import lightgbm as lgb

# новое ТЗ: отрицательный остаток = резерв, то есть свободного товара нет -> считаем «нет товара» при остатке <= 0
fp = ROOT / "src" / "features.py"
txt = fp.read_text(encoding="utf-8")
if "(sw == 0)" in txt:
    fp.write_text(txt.replace("(sw == 0)", "(sw <= 0)"), encoding="utf-8")
from src import features as _features
importlib.reload(_features)

SEED = 42
N_JOBS = 4          # не меняй число потоков между запусками


def build_features(c):
    return _features.build_features(Q0, ST, c, first_idx, wh_code, cat_code)


def make_training_set(last_cut, step=7):
    """Цель: средние недельные продажи за 9 недель после даты отсечки cc (cc всегда воскресенье)."""
    parts_X, parts_y = [], []
    for cc in range(last_cut, 179, -step):
        parts_X.append(build_features(cc))
        parts_y.append(Q0[:, cc + 1:cc + 1 + HORIZON].sum(axis=1) / WEEKS)
    X = pd.concat(parts_X, ignore_index=True)
    for col in ("warehouse", "category"):
        X[col] = X[col].astype("category")
    return X, np.concatenate(parts_y)


PARAMS = dict(
    objective="tweedie", tweedie_variance_power=1.3,
    n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=SEED, deterministic=True, force_row_wise=True, n_jobs=N_JOBS, verbose=-1,
)


def fit_rate(c_known):
    """Учим на датах, цель которых закончилась не позже c_known; прогноз недельной интенсивности на c_known."""
    X, y = make_training_set(c_known - HORIZON)
    model = lgb.LGBMRegressor(**PARAMS).fit(X, y)
    return model, np.clip(model.predict(build_features(c_known)), 0, None), len(X)


print("Определения готовы. Признаков:", build_features(D - 1).shape[1])

Определения готовы. Признаков: 22


In [5]:
# ЭТАП W4. Финальное обучение и submission (9 недель, первая неделя 2026-07-27)
FINAL_THRESHOLD = 1.0      # недельный прогноз ниже порога превращается в 0
FINAL_SCALE = 1.0          # множитель на прогноз
CHECK_DETERMINISM = True

t0 = time.time()
c_final = D - 1                                    # воскресенье 2026-07-26
X_fin, y_fin = make_training_set(c_final - HORIZON)
X_now = build_features(c_final)
print(f"Обучающая выборка: {len(X_fin):,} строк, подготовка за {time.time() - t0:.0f} c")


def train_and_predict():
    model = lgb.LGBMRegressor(**PARAMS).fit(X_fin, y_fin)
    return model, np.clip(model.predict(X_now), 0, None)


t1 = time.time()
final_model, rate_final = train_and_predict()
print(f"Модель обучена за {time.time() - t1:.0f} c")
if CHECK_DETERMINISM:
    _, rate_again = train_and_predict()
    print("Детерминизм (два обучения дали одинаковый результат):",
          "ДА" if np.array_equal(rate_final, rate_again) else "НЕТ")

rate_post = rate_final * FINAL_SCALE
rate_post = np.where(rate_post < FINAL_THRESHOLD, 0.0, rate_post)

weeks = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=WEEKS, freq="7D")   # понедельники
sub = pd.DataFrame({
    "week": np.tile(weeks.strftime("%Y-%m-%d"), S),
    "warehouse_name": np.repeat(meta["warehouse_name"].astype(str).values, WEEKS),
    "article": np.repeat(meta["article"].astype(str).values, WEEKS),
    "quantity": np.round(np.repeat(rate_post, WEEKS), 4),
})


def check_submission(df):
    problems = []
    if set(["week", "warehouse_name", "article", "quantity"]) - set(df.columns):
        return ["нет нужных колонок"]
    if df["quantity"].isna().any():
        problems.append("есть NaN в quantity")
    if (df["quantity"] < 0).any():
        problems.append("есть отрицательные значения")
    wk = pd.to_datetime(df["week"])
    if not (wk.dt.weekday == 0).all():
        problems.append("не все недели начинаются с понедельника")
    uw = np.sort(wk.unique())
    if len(uw) != WEEKS:
        problems.append(f"недель {len(uw)}, ожидалось {WEEKS}")
    elif (pd.Series(uw).diff().dropna() != pd.Timedelta(days=7)).any():
        problems.append("недели идут не подряд")
    if str(pd.Timestamp(uw[0]).date()) != "2026-07-27":
        problems.append("первая неделя не 2026-07-27")
    if df.duplicated(["week", "warehouse_name", "article"]).any():
        problems.append("есть дубли ключа")
    if len(df) != S * WEEKS:
        problems.append(f"строк {len(df)}, ожидалось {S * WEEKS}")
    return problems


problems = check_submission(sub)
print("\nПроверка формата:", "ВСЁ В ПОРЯДКЕ" if not problems else problems)
last9 = Q0[:, c_final - HORIZON + 1:c_final + 1].sum()
print(f"Прогноз за 9 недель: {sub['quantity'].sum():,.0f} шт.; факт предыдущих 9 недель: {last9:,.0f} шт.; "
      f"отношение {sub['quantity'].sum() / last9:.2f}")
print(f"Серий, которым прогноз обнулён: {(rate_post == 0).sum()} из {S} ({(rate_post == 0).mean():.0%})")
print(f"Недели прогноза: {weeks[0].date()} — {weeks[-1].date()} ({len(sub):,} строк)")

if not problems:
    out_path = ROOT / "data" / "processed" / "submission_weekly.csv"
    sub.to_csv(out_path, index=False)
    print(f"\nФайл сохранён: {out_path} ({out_path.stat().st_size / 1e6:.1f} МБ)")
    display(sub.head(3))
else:
    print("\nФайл НЕ сохранён: исправь проблемы выше.")

Обучающая выборка: 896,478 строк, подготовка за 9 c
Модель обучена за 28 c
Детерминизм (два обучения дали одинаковый результат): ДА

Проверка формата: ВСЁ В ПОРЯДКЕ
Прогноз за 9 недель: 209,165 шт.; факт предыдущих 9 недель: 212,336 шт.; отношение 0.99
Серий, которым прогноз обнулён: 5200 из 8789 (59%)
Недели прогноза: 2026-07-27 — 2026-09-21 (79,101 строк)

Файл сохранён: C:\Users\nikit\sber-forecast\data\processed\submission_weekly.csv (4.0 МБ)


,week,warehouse_name,article,quantity
0,2026-07-27,warehouse_name_0,char_article_0,0.0
1,2026-08-03,warehouse_name_0,char_article_0,0.0
2,2026-08-10,warehouse_name_0,char_article_0,0.0


In [6]:
# ЭТАП W5. Проверка порога и масштаба на трёх окнах и на сезонном окне (аналог 27.07.2025)
c_s = int((pd.Timestamp("2025-07-27") - dates[0]).days)
truths_s = truth(c_s)
naive_s = compute_metrics(truths_s, baselines(c_s)["последняя неделя"])

rates, t0 = {}, time.time()
for k in range(N_WINDOWS):
    _, rates[k], n = fit_rate(window_cut(k))
    print(f"окно {k}: {n:,} строк, {time.time() - t0:.0f} c", flush=True)
_, rate_s, n = fit_rate(c_s)
print(f"сезонное: {n:,} строк, {time.time() - t0:.0f} c", flush=True)

TH = [0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
SC = [0.9, 1.0, 1.1, 1.2]
combos = [(a, b) for a in TH for b in SC]


def sc_for(rate, Y, naive):
    out = {}
    for th, s in combos:
        r = np.where(rate * s < th, 0.0, rate * s)
        out[(th, s)] = proxy_score(compute_metrics(Y, flat_w(r)), naive)
    return out


tab = {k: sc_for(rates[k], truths[k], naive_m[k]) for k in range(N_WINDOWS)}
honest = []
for held in range(N_WINDOWS):
    oth = [k for k in range(N_WINDOWS) if k != held]
    best = min(combos, key=lambda cb: np.mean([tab[k][cb] for k in oth]))
    honest.append(tab[held][best])
best_all = min(combos, key=lambda cb: np.mean([tab[k][cb] for k in range(N_WINDOWS)]))

print("\nМодель без обработки:", round(np.mean([tab[k][(0, 1.0)] for k in range(N_WINDOWS)]), 4))
print(f"Лучший порог/масштаб по 3 окнам: {best_all}, score "
      f"{np.mean([tab[k][best_all] for k in range(N_WINDOWS)]):.4f}; честный {np.mean(honest):.4f}")
print("\nscore по сочетаниям (среднее по 3 окнам), строки — порог, столбцы — масштаб:")
display(pd.DataFrame({s: [np.mean([tab[k][(t, s)] for k in range(N_WINDOWS)]) for t in TH] for s in SC},
                     index=TH).round(4))

ts = sc_for(rate_s, truths_s, naive_s)
print("\nСЕЗОННОЕ окно (авг–сен 2025): прогноз/факт по объёму:", round(rate_s.sum() * WEEKS / truths_s.sum(), 3))
display(pd.DataFrame({s: [ts[(t, s)] for t in TH] for s in SC}, index=TH).round(4))
for k in range(N_WINDOWS):
    print(f"окно {k} прогноз/факт по объёму:", round(rates[k].sum() * WEEKS / truths[k].sum(), 3))

окно 0: 817,377 строк, 60 c
окно 1: 738,276 строк, 116 c
окно 2: 659,175 строк, 166 c
сезонное: 439,450 строк, 203 c

Модель без обработки: 0.8977
Лучший порог/масштаб по 3 окнам: (1.0, 0.9), score 0.7933; честный 0.8016

score по сочетаниям (среднее по 3 окнам), строки — порог, столбцы — масштаб:


,0.9,1.0,1.1,1.2
0.00,0.8860,0.8977,0.9235,0.9463
0.25,0.8891,0.8999,0.9249,0.9483
0.50,0.8476,0.8713,0.9126,0.9457
0.75,0.7993,0.8115,0.8424,0.8778
1.00,0.7933,0.7985,0.8215,0.8500
1.50,0.8188,0.8133,0.8266,0.8451
2.00,0.8549,0.8447,0.8520,0.8626
3.00,0.9187,0.9036,0.9057,0.9084



СЕЗОННОЕ окно (авг–сен 2025): прогноз/факт по объёму: 0.857


,0.9,1.0,1.1,1.2
0.00,0.9095,0.9020,0.9057,0.9205
0.25,0.9173,0.9084,0.9107,0.9244
0.50,0.8746,0.8736,0.8857,0.9127
0.75,0.8439,0.8321,0.8340,0.8500
1.00,0.8485,0.8297,0.8242,0.8344
1.50,0.8775,0.8552,0.8420,0.8447
2.00,0.9171,0.8871,0.8696,0.8681
3.00,0.9730,0.9477,0.9292,0.9232


окно 0 прогноз/факт по объёму: 0.906
окно 1 прогноз/факт по объёму: 1.036
окно 2 прогноз/факт по объёму: 1.137


In [7]:
# ЭТАП W6. Вариант «медиана»: модель предсказывает типичную (медианную) неделю, а не среднюю
MEDIAN_THRESHOLD = 1.0     # недельный прогноз ниже порога превращается в 0
MEDIAN_SCALE = 1.1         # медианная модель слегка занижает, поэтому множитель 1.1

c_final = D - 1                                    # воскресенье 2026-07-26


def make_training_set_median(last_cut, step=7):
    """Цель: медиана недельных продаж за 9 недель после даты отсечки (для редких товаров это 0)."""
    parts_X, parts_y = [], []
    for cc in range(last_cut, 179, -step):
        parts_X.append(build_features(cc))
        weekly = Q0[:, cc + 1:cc + 1 + HORIZON].reshape(S, WEEKS, 7).sum(axis=2)
        parts_y.append(np.median(weekly, axis=1))
    X = pd.concat(parts_X, ignore_index=True)
    for col in ("warehouse", "category"):
        X[col] = X[col].astype("category")
    return X, np.concatenate(parts_y)


PARAMS_MED = {k: v for k, v in PARAMS.items() if k != "tweedie_variance_power"}
PARAMS_MED["objective"] = "l1"                     # L1 = модель ищет медиану

t0 = time.time()
X_med, y_med = make_training_set_median(c_final - HORIZON)
X_now_med = build_features(c_final)
print(f"Обучающая выборка: {len(X_med):,} строк, подготовка за {time.time() - t0:.0f} c")


def train_and_predict_median():
    model = lgb.LGBMRegressor(**PARAMS_MED).fit(X_med, y_med)
    return model, np.clip(model.predict(X_now_med), 0, None)


t1 = time.time()
model_med, rate_med = train_and_predict_median()
print(f"Модель обучена за {time.time() - t1:.0f} c")
_, rate_med_again = train_and_predict_median()
print("Детерминизм (два обучения дали одинаковый результат):",
      "ДА" if np.array_equal(rate_med, rate_med_again) else "НЕТ")

rate_med_post = rate_med * MEDIAN_SCALE
rate_med_post = np.where(rate_med_post < MEDIAN_THRESHOLD, 0.0, rate_med_post)

weeks = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=WEEKS, freq="7D")
sub_med = pd.DataFrame({
    "week": np.tile(weeks.strftime("%Y-%m-%d"), S),
    "warehouse_name": np.repeat(meta["warehouse_name"].astype(str).values, WEEKS),
    "article": np.repeat(meta["article"].astype(str).values, WEEKS),
    "quantity": np.round(np.repeat(rate_med_post, WEEKS), 4),
})

problems = check_submission(sub_med)            # функция из ячейки W4
print("\nПроверка формата:", "ВСЁ В ПОРЯДКЕ" if not problems else problems)
last9 = Q0[:, c_final - HORIZON + 1:c_final + 1].sum()
print(f"Прогноз за 9 недель: {sub_med['quantity'].sum():,.0f} шт.; факт предыдущих 9 недель: {last9:,.0f} шт.; "
      f"отношение {sub_med['quantity'].sum() / last9:.2f}")
print(f"Серий, которым прогноз обнулён: {(rate_med_post == 0).sum()} из {S} ({(rate_med_post == 0).mean():.0%})")

# сравнение с вариантом из W4 (средняя модель)
zero_a, zero_b = (rate_post == 0), (rate_med_post == 0)
print(f"Совпадение с вариантом W4: обнулены в обоих {int((zero_a & zero_b).sum())}, "
      f"только в W4 {int((zero_a & ~zero_b).sum())}, только в W6 {int((~zero_a & zero_b).sum())}")

if not problems:
    out_path = ROOT / "data" / "processed" / "submission_median.csv"
    sub_med.to_csv(out_path, index=False)
    print(f"\nФайл сохранён: {out_path} ({out_path.stat().st_size / 1e6:.1f} МБ)")
else:
    print("\nФайл НЕ сохранён: исправь проблемы выше.")

Обучающая выборка: 896,478 строк, подготовка за 15 c
Модель обучена за 66 c
Детерминизм (два обучения дали одинаковый результат): ДА

Проверка формата: ВСЁ В ПОРЯДКЕ
Прогноз за 9 недель: 209,636 шт.; факт предыдущих 9 недель: 212,336 шт.; отношение 0.99
Серий, которым прогноз обнулён: 5223 из 8789 (59%)
Совпадение с вариантом W4: обнулены в обоих 5092, только в W4 108, только в W6 131

Файл сохранён: C:\Users\nikit\sber-forecast\data\processed\submission_median.csv (4.0 МБ)


In [8]:
# ЭТАП W7. Несколько вариантов прогноза из уже обученных моделей (для отправки организатору и сравнения по лидерборду)
VARIANTS = {
    # имя файла : (какая модель, порог, масштаб)
    "submission_A_mean_t1.0_s1.0":   ("mean",   1.0, 1.0),
    "submission_B_median_t1.0_s1.1": ("median", 1.0, 1.1),
    "submission_C_median_t0.5_s1.2": ("median", 0.5, 1.2),
    "submission_D_mean_t0.75_s1.1":  ("mean",   0.75, 1.1),
}
RATES = {"mean": rate_final, "median": rate_med}          # из ячеек W4 и W6
weeks = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=WEEKS, freq="7D")

summary_rows = []
for name, (kind, th, sc) in VARIANTS.items():
    r = RATES[kind] * sc
    r = np.where(r < th, 0.0, r)
    sub_v = pd.DataFrame({
        "week": np.tile(weeks.strftime("%Y-%m-%d"), S),
        "warehouse_name": np.repeat(meta["warehouse_name"].astype(str).values, WEEKS),
        "article": np.repeat(meta["article"].astype(str).values, WEEKS),
        "quantity": np.round(np.repeat(r, WEEKS), 4),
    })
    problems = check_submission(sub_v)                     # функция из ячейки W4
    if not problems:
        sub_v.to_csv(ROOT / "data" / "processed" / f"{name}.csv", index=False)
    summary_rows.append({
        "файл": name + ".csv", "формат": "ок" if not problems else str(problems),
        "объём за 9 нед.": round(float(sub_v["quantity"].sum())),
        "обнулено серий": f"{(r == 0).mean():.0%}",
    })
display(pd.DataFrame(summary_rows))

,файл,формат,объём за 9 нед.,обнулено серий
0,submission_A_mean_t1.0_s1.0.csv,ок,209165,59%
1,submission_B_median_t1.0_s1.1.csv,ок,209636,59%
2,submission_C_median_t0.5_s1.2.csv,ок,231595,55%
3,submission_D_mean_t0.75_s1.1.csv,ок,236803,50%
